# 02 — Model A: default probability

Dataset: `credit_risk_dataset.csv`
Target: `SeriousDlqin2yrs`
Success: **AUC-ROC ≥ 0.80**, **KS ≥ 0.35**

Flow: baseline logistic regression → Optuna-tuned XGBoost → isotonic calibration → SHAP → save `ml/models/model_a_default.pkl`.

Reads pre-split data from `data/processed/` — run `python -m ml.training.prepare_data` first if those files don't exist yet.


In [ ]:
%matplotlib inline


In [3]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "ml" / "pipeline" / "features.py").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ml.config import (
    DATA_PROCESSED,
    DATA_RAW,
    FIGURES_DIR,
    MODELS_DIR,
    RANDOM_STATE,
    REPORTS_DIR,
    ensure_dirs,
)

ensure_dirs()
print("Project root:", ROOT)


Project root: C:\Users\Dilini\OneDrive\Documents\IJSE Doc\Machine Learning\ML Project


In [ ]:
import json
from datetime import datetime, timezone

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ks_2samp
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
    roc_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")


def ks_statistic(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    pos, neg = y_score[y_true == 1], y_score[y_true == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan")
    return float(ks_2samp(pos, neg).statistic)


def classification_metrics(y_true, y_proba, threshold=0.5):
    y_pred = (np.asarray(y_proba) >= threshold).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "ks_statistic": ks_statistic(y_true, y_proba),
        "average_precision": float(average_precision_score(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(report["1"]["precision"]),
        "recall": float(report["1"]["recall"]),
        "classification_report": report,
    }


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mape": float(np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1, None))) * 100),
    }


def stratified_split(X, y, random_state=RANDOM_STATE):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def save_pipeline(pipe, name, metadata):
    path = MODELS_DIR / f"{name}.pkl"
    joblib.dump(pipe, path)
    metadata = {**metadata, "artifact": str(path.as_posix()), "saved_at": datetime.now(timezone.utc).isoformat()}
    (MODELS_DIR / f"{name}.meta.json").write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")
    print("Saved", path)
    return path


## Load pre-split data (from `ml.training.prepare_data`)


In [ ]:
from ml.pipeline.features import DEFAULT_FEATURE_LABELS, DEFAULT_OUTPUT_COLS, DefaultRiskFeatures

X_train = pd.read_parquet(DATA_PROCESSED / "a_train.parquet")
X_val = pd.read_parquet(DATA_PROCESSED / "a_val.parquet")
X_test = pd.read_parquet(DATA_PROCESSED / "a_test.parquet")
y_train = pd.read_parquet(DATA_PROCESSED / "ya_train.parquet").iloc[:, 0]
y_val = pd.read_parquet(DATA_PROCESSED / "ya_val.parquet").iloc[:, 0]
y_test = pd.read_parquet(DATA_PROCESSED / "ya_test.parquet").iloc[:, 0]

scale_pos_weight = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
print(f"train/val/test = {len(X_train)}/{len(X_val)}/{len(X_test)}")
print(f"default rate (train) = {y_train.mean():.4f}   scale_pos_weight = {scale_pos_weight:.2f}")


## Baseline — LogisticRegression (`class_weight='balanced'`)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

baseline = Pipeline([
    ("features", DefaultRiskFeatures()),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)),
])
baseline.fit(X_train, y_train)
baseline_metrics = classification_metrics(y_test, baseline.predict_proba(X_test)[:, 1])
print(f"Baseline  AUC={baseline_metrics['auc_roc']:.4f}  KS={baseline_metrics['ks_statistic']:.4f}  F1={baseline_metrics['f1']:.4f}")


## Shared features + Optuna XGBoost

Hyperparameter search uses a 50k stratified sample of train for speed; the final booster is fit on **all** of train. Change `N_TRIALS` if you want a shorter run.


In [ ]:
from sklearn.model_selection import train_test_split as _tts
from xgboost import XGBClassifier
import optuna
from sklearn.metrics import roc_auc_score

N_TRIALS = 20  # drop to 8 for a faster demo

features = DefaultRiskFeatures().fit(X_train)
X_train_f, X_val_f, X_test_f = features.transform(X_train), features.transform(X_val), features.transform(X_test)

if len(X_train_f) > 50_000:
    X_search, _, y_search, _ = _tts(X_train_f, y_train, train_size=50_000, stratify=y_train, random_state=RANDOM_STATE)
else:
    X_search, y_search = X_train_f, y_train

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 12),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
        "scale_pos_weight": scale_pos_weight,
        "max_delta_step": 1,
        "tree_method": "hist",
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
        "eval_metric": "auc",
        "early_stopping_rounds": 40,
    }
    model = XGBClassifier(**params)
    model.fit(X_search, y_search, eval_set=[(X_val_f, y_val)], verbose=False)
    return float(roc_auc_score(y_val, model.predict_proba(X_val_f)[:, 1]))

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
best_params = study.best_params
print(f"Best val AUC={study.best_value:.4f}")
best_params


## Final booster + isotonic calibration (sklearn 1.8: `FrozenEstimator`)


In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

xgb = XGBClassifier(
    **best_params,
    scale_pos_weight=scale_pos_weight,
    max_delta_step=1,
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    eval_metric="auc",
    early_stopping_rounds=40,
)
xgb.fit(X_train_f, y_train, eval_set=[(X_val_f, y_val)], verbose=False)
raw_test = classification_metrics(y_test, xgb.predict_proba(X_test_f)[:, 1])
print(f"Raw XGB   AUC={raw_test['auc_roc']:.4f}  KS={raw_test['ks_statistic']:.4f}")

calibrated = CalibratedClassifierCV(estimator=FrozenEstimator(xgb), method="isotonic", cv=3)
calibrated.fit(X_val_f, y_val)
test_metrics = classification_metrics(y_test, calibrated.predict_proba(X_test_f)[:, 1])
print(f"Calibrated AUC={test_metrics['auc_roc']:.4f}  KS={test_metrics['ks_statistic']:.4f}  Brier={test_metrics['brier']:.4f}")
print("Meets AUC target:", test_metrics["auc_roc"] >= 0.80, "| Meets KS target:", test_metrics["ks_statistic"] >= 0.35)
print(classification_report(y_test, (calibrated.predict_proba(X_test_f)[:, 1] >= 0.5).astype(int), zero_division=0))


## ROC, calibration, SHAP

The 0.5 class cutoff looks weak on recall because only ~7% default. The API should use **calibrated probability** and risk bands (Low < 0.10, Medium < 0.35, else High), not a 0.5 label.


In [ ]:
pipe = Pipeline([("features", features), ("model", calibrated)])
proba_test = pipe.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fpr, tpr, _ = roc_curve(y_test, proba_test)
axes[0].plot(fpr, tpr, label=f"AUC={roc_auc_score(y_test, proba_test):.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_title("ROC — Model A")
axes[0].legend()
frac, mean_pred = calibration_curve(y_test, proba_test, n_bins=10, strategy="quantile")
axes[1].plot(mean_pred, frac, marker="o")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_title("Calibration — Model A")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "model_a_roc.png", dpi=140)
fig.savefig(FIGURES_DIR / "model_a_calibration.png", dpi=140)
plt.show()


In [ ]:
import shap

sample = X_test_f.sample(n=min(200, len(X_test_f)), random_state=1)
labels = [DEFAULT_FEATURE_LABELS.get(c, c) for c in sample.columns]
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(sample)
shap.summary_plot(shap_values, sample, feature_names=labels, show=False, max_display=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_summary_model_a.png", dpi=140, bbox_inches="tight")
plt.show()


## Package for the API


In [ ]:
save_pipeline(
    pipe,
    "model_a_default",
    {
        "model_version": "model_a_v1",
        "task": "default_probability",
        "target": "SeriousDlqin2yrs",
        "features": DEFAULT_OUTPUT_COLS,
        "best_params": best_params,
        "scale_pos_weight": scale_pos_weight,
        "metrics": {
            "baseline_auc": baseline_metrics["auc_roc"],
            "baseline_ks": baseline_metrics["ks_statistic"],
            "tuned_auc": test_metrics["auc_roc"],
            "tuned_ks": test_metrics["ks_statistic"],
            "tuned_brier": test_metrics["brier"],
        },
        "risk_bands": {"low_lt": 0.10, "medium_lt": 0.35, "high_ge": 0.35},
    },
)

# smoke test: one raw row through the saved pipeline
loaded = joblib.load(MODELS_DIR / "model_a_default.pkl")
print("sample default probability:", float(loaded.predict_proba(X_test.head(1))[:, 1][0]))
